In [ ]:
# Read and create animated side-by-side histograms for time-series data
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.mixture import GaussianMixture
from scipy.stats import norm
import matplotlib.animation as animation
from IPython.display import HTML

# Relocated to src/rarefied/io/histograms.py (docs/CLEANUP_TODO.md, Phase 4).
# Function bodies moved verbatim -- see that module for the actual implementations.
from rarefied.io.histograms import (
    read_all_timesteps_histogram_2D,
    read_all_timesteps_histogram_3D,
)




def create_histogram_animation_from_data(x_data, y_data, output_video_path=None, fps=10, dpi=100, progress_callback=None):
    """
    Create side-by-side animated histograms from pre-loaded histogram data.
    
    Parameters:
    - x_data: Pre-loaded X velocity histogram data dictionary (from read_all_timesteps_histogram)
    - y_data: Pre-loaded Y velocity histogram data dictionary (from read_all_timesteps_histogram)
    - output_video_path: Path to save video (if None, just display animation)
    - fps: Frames per second for video (default: 10)
    - dpi: DPI for video output (default: 100)
    - progress_callback: Optional callback function(frame, total_frames, status_str) for progress updates
    
    Returns:
    - ani: Animation object
    """
    print(f"X velocity: {len(x_data)} timesteps loaded")
    print(f"Y velocity: {len(y_data)} timesteps loaded")
    
    # Get sorted timestep indices
    timesteps = sorted(set(list(x_data.keys()) + list(y_data.keys())))
    print(f"Total timesteps to animate: {len(timesteps)}")
    
    # Pre-calculate y-axis limits to avoid recalculating every frame
    print("Pre-calculating axis limits...")
    y_max_x = max([max(x_data[ts]['bin_normalized']) for ts in timesteps if ts in x_data]) if x_data else 1
    y_max_y = max([max(y_data[ts]['bin_normalized']) for ts in timesteps if ts in y_data]) if y_data else 1
    
    # Helper function to report progress
    def report_progress(frame_idx, total_frames, stage="rendering"):
        if progress_callback is not None:
            percentage = (frame_idx / total_frames) * 100
            status_str = f"[{stage}] Frame {frame_idx + 1}/{total_frames} ({percentage:.1f}%)"
            progress_callback(frame_idx + 1, total_frames, status_str)
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Initialize stairs lines and titles
    stairs1 = None
    stairs2 = None
    title1 = None
    title2 = None
    
    def animate(frame_idx):
        nonlocal stairs1, stairs2, title1, title2
        
        timestep = timesteps[frame_idx]
        
        # Report progress
        report_progress(frame_idx, len(timesteps), stage="rendering")
        
        # Get data for this timestep
        x_ts_data = x_data.get(timestep)
        y_ts_data = y_data.get(timestep)
        
        # First frame: Create stairs plots
        if stairs1 is None:
            # Plot X velocity histogram
            if x_ts_data is not None:
                bin_coords = x_ts_data['bin_coords']
                bin_normalized = x_ts_data['bin_normalized']
                header = x_ts_data['header']
                
                # Extend edges for stairs plot (needs len(values) + 1 edges)
                if len(bin_coords) > 0:
                    spacing = bin_coords[-1] - bin_coords[-2] if len(bin_coords) > 1 else bin_coords[-1]
                    edges = bin_coords + [bin_coords[-1] + spacing]
                else:
                    edges = bin_coords
                
                stairs1 = ax1.stairs(bin_normalized, edges, 
                                    color='steelblue', linewidth=1.5, alpha=0.8, fill=True)
                ax1.set_xlabel('Velocity (units)', fontsize=11)
                ax1.set_ylabel('Normalized Count', fontsize=11)
                ax1.set_ylim(0, 0.04)
                ax1.set_xlim(-5000, 5000)
                ax1.grid(True, alpha=0.3)
                title1 = ax1.set_title('')
        
        # First frame: Create stairs plot for Y
        if stairs2 is None:
            if y_ts_data is not None:
                bin_coords = y_ts_data['bin_coords']
                bin_normalized = y_ts_data['bin_normalized']
                header = y_ts_data['header']
                
                # Extend edges for stairs plot (needs len(values) + 1 edges)
                if len(bin_coords) > 0:
                    spacing = bin_coords[-1] - bin_coords[-2] if len(bin_coords) > 1 else bin_coords[-1]
                    edges = bin_coords + [bin_coords[-1] + spacing]
                else:
                    edges = bin_coords
                
                stairs2 = ax2.stairs(bin_normalized, edges, 
                                    color='coral', linewidth=1.5, alpha=0.8, fill=True)
                ax2.set_xlabel('Velocity (units)', fontsize=11)
                ax2.set_ylabel('Normalized Count', fontsize=11)
                ax2.set_ylim(0, 0.04)
                ax2.set_xlim(-5000, 5000)
                ax2.grid(True, alpha=0.3)
                title2 = ax2.set_title('')
        
        # Update stairs heights (much faster than recreating)
        if stairs1 is not None and x_ts_data is not None:
            bin_normalized = x_ts_data['bin_normalized']
            header = x_ts_data['header']
            
            stairs1.set_data(bin_normalized)
            title1.set_text(f'X Velocity - TimeStep {timestep}\nTotal Counts: {header["Total_counts"]:.2e}')
        
        if stairs2 is not None and y_ts_data is not None:
            bin_normalized = y_ts_data['bin_normalized']
            header = y_ts_data['header']
            
            stairs2.set_data(bin_normalized)
            title2.set_text(f'Y Velocity - TimeStep {timestep}\nTotal Counts: {header["Total_counts"]:.2e}')
        
        fig.suptitle(f'Histogram Time Series - Frame {frame_idx + 1}/{len(timesteps)}', 
                    fontsize=14, fontweight='bold', y=1.00)
        
        return [stairs1, stairs2, title1, title2] if stairs1 and stairs2 else []
    
    # Create animation
    ani = animation.FuncAnimation(fig, animate, frames=len(timesteps), 
                                 interval=1000/fps, repeat=True, blit=False)
    
    # Save video if path provided
    if output_video_path is not None:
        print(f"\nSaving animation to: {output_video_path}")
        print("This may take a few minutes...")
        
        if progress_callback is not None:
            progress_callback(0, len(timesteps), "[saving] Starting video encoding...")
        
        try:
            writer = animation.FFMpegWriter(fps=fps, metadata=dict(artist='Meteor Modelling'),
                                           bitrate=1800)
            ani.save(output_video_path, writer=writer)
            
            if progress_callback is not None:
                progress_callback(len(timesteps), len(timesteps), "[saving] Video encoding complete!")
            
            print(f"✓ Video saved successfully!")
        except Exception as e:
            print(f"Error saving video: {e}")
            print("Make sure FFmpeg is installed on your system")
    
    plt.show()
    return ani

def create_histogram_animation(x_filepath, y_filepath, output_video_path=None, fps=10, dpi=100, progress_callback=None):
    """
    Create side-by-side animated histograms for X and Y velocity data.
    Reads data from files and creates animation.
    
    Parameters:
    - x_filepath: Path to X velocity histogram file
    - y_filepath: Path to Y velocity histogram file
    - output_video_path: Path to save video (if None, just display animation)
    - fps: Frames per second for video (default: 10)
    - dpi: DPI for video output (default: 100)
    - progress_callback: Optional callback function(frame, total_frames, status_str) for progress updates
    
    Returns:
    - ani: Animation object
    """
    print("Reading X velocity data...")
    x_data = read_all_timesteps_histogram(x_filepath)
    print(f"  Loaded {len(x_data)} timesteps")
    
    print("Reading Y velocity data...")
    y_data = read_all_timesteps_histogram(y_filepath)
    print(f"  Loaded {len(y_data)} timesteps")
    
    # Call the data-based animation function
    return create_histogram_animation_from_data(x_data, y_data, output_video_path, fps, dpi, progress_callback)

# Example usage with progress callback:
def progress_monitor(frame, total_frames, status_str):
    """Simple progress callback that prints status updates."""
    # Print every 10 frames to avoid cluttering output
    if frame % 10 == 0 or frame == total_frames:
        print(f"  {status_str}")

# data_3D_ORIGINAL = read_all_timesteps_histogram_3D(Path(f"/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p1_3D/0/4_3D.histo"))
# print("Done!")

# data_2D_ORIGINAL = read_all_timesteps_histogram_2D(Path(f"/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p1/0/1.histo"))
# print("Done!")

# # data_2D_HIGHER_KN = read_all_timesteps_histogram_2D(Path(f"/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/ARGON_CYLINDER_VALIDATION_2D_KN_eq_0p1/0/5.histo"))
# # print("Done!")

# Tdata = data_3D_ORIGINAL[50000]['Tdata']
# Ndata = data_3D_ORIGINAL[50000]['Ndata']
# Zdata = data_3D_ORIGINAL[50000]['Zdata']

# print(Tdata[0,0,:])  # should vary in Z
# print(Ndata[0,:,0])  # should vary in N
# print(Zdata[:,0,0])  # should vary in T

# joint_mpf = data_3D_ORIGINAL[50000]['joint_pmf']
# print("Xdata:", Tdata[0:5,0:5,0])
# print("Ydata:", Ndata[0:5,0:5,0])
# print("Zdata:", Zdata[0:5,0:5,0:5])

# z_eq_0_ind = Zdata.shape[2] // 2 
# fig = plt.figure(figsize=(8,6))
# ax = fig.add_subplot(121, projection='3d')
# # surf = ax.plot_surface(X,Y,data_2D_ORIGINAL[50000]['bin_normalized'], cmap='viridis')
# # ax.view_init(elev=0, azim=90)
# ctr = ax.plot_surface(Tdata[:,:,z_eq_0_ind], Ndata[:,:,z_eq_0_ind], joint_mpf[:,:,z_eq_0_ind], cmap='viridis')
# # fig.colorbar(ctr)
# # ax.set_xlim(-4000,4000)
# # ax.set_ylim(-4000,4000)
# ax.set_title('Cylinder 2D Distribution: Original Kn=0.01')
# ax.set_xlabel('Tangential Velocity (units)', fontsize=11)
# ax.set_ylabel('Normal Velocity (units)', fontsize=11)
# # ax.set_box_aspect(1)
# # plt.show()

# ax = fig.add_subplot(122)
# idx = Ndata.shape[1] // 2 
# plt.plot(Ndata[:,idx], joint_mpf[:,idx], label="true")
# plt.xlabel('Normal Velocity (units)', fontsize=11)
# plt.ylabel('Normalized Count', fontsize=11)
# plt.title('Normal Velocity Distribution at Tangential Velocity=0', fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.legend()
# plt.show()

# print(np.sum(data_2D_ORIGINAL[40000]['bin_normalized']))

# fig, ax = plt.subplots()
# # surf = ax.plot_surface(X,Y,data_2D[50000]['bin_normalized'], cmap='viridis')
# ctr = ax.contourf(X, Y, data_2D_HIGHER_KN[50000]['bin_normalized'], levels=20, cmap='viridis')
# fig.colorbar(ctr)
# ax.set_title('Cylinder 2D Distribution: Higher Kn=0.1')
# ax.set_xlabel('X Velocity (units)', fontsize=11)
# ax.set_ylabel('Y Velocity (units)', fontsize=11)
# # ax.set_xlim(-4000,4000)
# # ax.set_ylim(-4000,4000)
# plt.show()

In [ ]:
# Region tangent vectors and angles from metadata
from pathlib import Path
import glob
import numpy as np
import os
import pandas as pd

# Relocated to src/rarefied/io/sparta_runs.py (docs/CLEANUP_TODO.md, Phase 4).
# Function bodies moved verbatim -- see that module for the actual implementations.
from rarefied.io.sparta_runs import load_all_histograms_from_run_2D, read_region_metadata




path = "/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p01/0/"
metadata_path = Path("/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/validation_case_argon_circle/argon_circle_0_metadata.txt")

# region_df, data = load_all_histograms_from_run_2D(path, metadata_path)

# import pickle
# # To SAVE
# with open('KN_eq_0p01_data.pkl', 'wb') as f:
#     pickle.dump((region_df, data), f)

# # To LOAD
# with open('my_data.pkl', 'rb') as f:
#     region_df, data = pickle.load(f)

In [ ]:
# Relocated to src/rarefied/moments/hermite.py and src/rarefied/reconstruct/flux.py
# (docs/CLEANUP_TODO.md, Phase 3). Function bodies moved verbatim -- see those
# modules for the actual implementations; this cell only re-imports them so the
# notebook's existing cells (5, 6) keep working unchanged.
from rarefied.moments.hermite import (
    fit_hermite_distribution_2d,
    fit_hermite_distribution_3d,
    get_mean_var_2D,
    get_mean_var_3D,
    hermite_polynomial,
    pdf_hermite_2d,
    pdf_hermite_3d,
)
from rarefied.reconstruct.flux import (
    Q_n_p,
    Q_n_p_half,
    energy_flux_from_3D_distribution,
    energy_flux_from_3D_hermite_fit,
    fast_heat_flux_from_3D_hermite_fit,
    fast_momentum_flux_from_3D_hermite_fit,
    heat_flux_from_2D_distribution,
    heat_flux_from_2D_hermite_fit,
    heat_flux_from_3D_distribution,
    heat_flux_from_3D_hermite_fit,
    kinetic_energy_flux_from_2D,
    momentum_flux_from_2D_hermite_fit,
    momentum_flux_from_2d_distribution,
    momentum_flux_from_3D_hermite_fit,
    momentum_flux_from_3d_distribution,
)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ------------------------------------------------------------------
# Parsers for SPARTA ITEM tables (SURFS / CELLS)
# ------------------------------------------------------------------
# Relocated to src/rarefied/io/sparta_runs.py (docs/CLEANUP_TODO.md, Phase 4).
# Function bodies moved verbatim -- see that module for the actual implementations.
from rarefied.io.sparta_runs import (
    load_run_data_3D,
    parse_sparta_item_table,
    relabel_grid_columns,
    relabel_surface_columns,
)









def parity_plot(ax, x, y, xlabel, ylabel, title):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    ax.scatter(x, y, s=50, alpha=0.85)

    if len(x) > 0:
        lo = min(np.min(x), np.min(y))
        hi = max(np.max(x), np.max(y))
        if hi > lo:
            ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.2, label="y = x")

    if len(x) > 1:
        corr = np.corrcoef(x, y)[0, 1]
        r2 = corr**2
    else:
        r2 = np.nan

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}\n$R^2$={r2:.3f}" if np.isfinite(r2) else title)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")



# ------------------------------------------------------------------
# One-time data loading cache (IO-heavy step)
# ------------------------------------------------------------------
# region_cache = load_run_data("/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p01/0/")
# data_dir = Path("/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p01/0")
# region_ids = sorted({
#     int(p.stem)
#     for p in data_dir.glob("*.histo")
#     if p.stem.isdigit()
# })
# print("Region IDs found:", region_ids)
# n_terms_max = 15
# fit_window = (-2000, 2000)

# region_cache = {}

# for region in tqdm(region_ids, desc="Processing regions"):
#     hist_file = data_dir / f"{region}.histo"
#     surf_file = data_dir / f"out_region{region}surf.50000"
#     grid_file = data_dir / f"out_region{region}grid.50000"

#     if not (hist_file.exists() and surf_file.exists() and grid_file.exists()):
#         continue

#     data = read_all_timesteps_histogram_2D(hist_file)

#     common_ts = sorted(set(data.keys()))
#     if not common_ts:
#         continue

#     ts = common_ts[-1]

#     grid_df = relabel_grid_columns(parse_sparta_item_table(grid_file, "CELLS"))
#     surf_df = relabel_surface_columns(parse_sparta_item_table(surf_file, "SURFS"))

#     region_cache[region] = {
#         "ts": ts,
#         "data": data[ts],
#         "grid_df": grid_df,
#         "surf_df": surf_df,
#     }

# print(f"Loaded cache for {len(region_cache)} regions (IO complete).")

In [ ]:
# Relocated to src/rarefied/moments/pipeline.py (docs/CLEANUP_TODO.md, Phase 3).
# Function body moved verbatim -- see that module for the actual implementation.
from rarefied.moments.pipeline import calculate_and_compare_moments


In [7]:
# Cp/Cf scatter plots from stress lists using user-defined normalization
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

def plot_cp_cf_ch(moments, m, n_inf, u_inf, title=None, all_in_one_plot=False):
    pn_model = moments['pn_model']
    pt_model = moments['pt_model']
    pn_dist = moments['pn_dist']
    pt_dist = moments['pt_dist']
    pn_dsmc = moments['pn_dsmc']
    pt_dsmc = moments['pt_dsmc']
    e_model = moments['e_model']
    e_dsmc = moments['e_dsmc']
    e_dist = moments['e_dist']

    p_inf = 0.5 *m * n_inf * u_inf**2 # free stream dynamic pressure
    q_inf = 0.5 * m * n_inf * u_inf**3 # free stream heat flux

    cp_model = pn_model / p_inf
    cp_dsmc = pn_dsmc / p_inf
    cp_dist = pn_dist / p_inf
    cf_model = pt_model / p_inf
    cf_dsmc = pt_dsmc / p_inf
    cf_dist = pt_dist / p_inf
    ch_model = e_model / q_inf
    ch_dsmc = e_dsmc / q_inf
    ch_dist = e_dist / q_inf
    
    region_idx = np.arange(len(cp_model))

    # Font Sizes
    plt.rcParams.update({
        'font.size': 14,          # General font size
        'axes.titlesize': 18,     # Title size
        'axes.labelsize': 16,     # Axis label (X and Y) size
        'xtick.labelsize': 12,    # Tick labels
        'ytick.labelsize': 12
    })

    # Cp plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111)
    ax.scatter(region_idx, cp_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cp Model')
    ax.scatter(region_idx, cp_dsmc, s=70, marker='x', facecolors='k', label='Cp DSMC')
    ax.scatter(region_idx, cp_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cp Distribution')
    # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_ylabel('Cp')
    ax.set_xlabel('Region Index')
    if title:
        ax.set_title(title)
    else:
        ax.set_title('Cp')
        
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman"]
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    # Cf plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111)
    ax.scatter(region_idx, cf_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cf Model')
    ax.scatter(region_idx, cf_dsmc, s=70, marker='x', facecolors='k', label='Cf DSMC')
    ax.scatter(region_idx, cf_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cf Distribution')
    # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_ylabel('Cf')
    ax.set_xlabel('Region Index')
    if title:
        ax.set_title(title)
    else:
        ax.set_title('Cf')
        
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman"]
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    # Ch plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111)
    ax.scatter(region_idx, ch_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='CH Model')
    ax.scatter(region_idx, ch_dsmc, s=70, marker='x', facecolors='k', label='CH DSMC')
    ax.scatter(region_idx, ch_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='CH Distribution')
    # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    
    ax.set_ylabel('CH')
    ax.set_xlabel('Region Index')
    if title:
        ax.set_title(title)
    else:
        ax.set_title('CH')
        
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman"]
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    if all_in_one_plot:

            # Font Sizes
        plt.rcParams.update({
            'font.size': 28,          # General font size
            'axes.titlesize': 36,     # Title size
            'axes.labelsize': 32,     # Axis label (X and Y) size
            'xtick.labelsize': 24,    # Tick labels
            'ytick.labelsize': 24
        })
        
        fig = plt.figure(figsize=(30, 8))
        ax1 = fig.add_subplot(131)
        
        

        # Cp plot
        ax1.scatter(region_idx, cp_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cp Model')
        ax1.scatter(region_idx, cp_dsmc, s=70, marker='x', facecolors='k', label='Cp DSMC')
        ax1.scatter(region_idx, cp_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cp Distribution')
        # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
        
        ax1.set_ylabel('Cp')
        ax1.set_xlabel('Region Index')
        # ax1.legend()
            
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = ["Times New Roman"]
        ax1.grid(True, alpha=0.25)

        # Cf plot
        ax2 = fig.add_subplot(132)
        ax2.scatter(region_idx, cf_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cf Model')
        ax2.scatter(region_idx, cf_dsmc, s=70, marker='x', facecolors='k', label='Cf DSMC')
        ax2.scatter(region_idx, cf_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cf Distribution')
        # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
        
        ax2.set_ylabel('Cf')
        ax2.set_xlabel('Region Index')
            
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = ["Times New Roman"]
        ax2.grid(True, alpha=0.25)

        # Ch plot
        ax3 = fig.add_subplot(133)
        ax3.scatter(region_idx, ch_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Model')
        ax3.scatter(region_idx, ch_dsmc, s=70, marker='x', facecolors='k', label='DSMC')
        ax3.scatter(region_idx, ch_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='CH Distribution')
        # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
        
        ax3.set_ylabel('CH')
        ax3.set_xlabel('Region Index')
            
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = ["Times New Roman"]
        ax3.grid(True, alpha=0.25)
        ax3.legend(
            fontsize=16,            # Specifically set a smaller size for this legend
            loc='upper right',      # Force it to the top right corner
            frameon=True,           # Optional: adds a box around the legend
            edgecolor='k'           # Optional: makes the box border black
        )

        if title:
            fig.suptitle(title, fontsize=18)
        else:
            fig.suptitle('Cp, Cf, CH Comparison', fontsize=18)
        plt.show()

def plot_cp_cf_ch_all(moment_list, m, n_inf, u_inf, title=None, all_in_one_plot=False):
    for i in range(len(moment_list)):

        moments = moment_list[i]

        pn_model = moments['pn_model']
        pt_model = moments['pt_model']
        pn_dist = moments['pn_dist']
        pt_dist = moments['pt_dist']
        pn_dsmc = moments['pn_dsmc']
        pt_dsmc = moments['pt_dsmc']
        e_model = moments['e_model']
        e_dsmc = moments['e_dsmc']
        e_dist = moments['e_dist']

        p_inf = 0.5 *m * n_inf * u_inf**2 # free stream dynamic pressure
        q_inf = 0.5 * m * n_inf * u_inf**3 # free stream heat flux

        cp_model = pn_model / p_inf
        cp_dsmc = pn_dsmc / p_inf
        cp_dist = pn_dist / p_inf
        cf_model = pt_model / p_inf
        cf_dsmc = pt_dsmc / p_inf
        cf_dist = pt_dist / p_inf
        ch_model = e_model / q_inf
        ch_dsmc = e_dsmc / q_inf
        ch_dist = e_dist / q_inf
    
        region_idx = np.arange(len(cp_model))

        # Font Sizes
        plt.rcParams.update({
            'font.size': 14,          # General font size
            'axes.titlesize': 18,     # Title size
            'axes.labelsize': 16,     # Axis label (X and Y) size
            'xtick.labelsize': 12,    # Tick labels
            'ytick.labelsize': 12
        })

        # Cp plot
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111)
        n_terms = (i+1)**2
        ax.scatter(region_idx, cp_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='n_terms={}'.format(n_terms))
        ax.scatter(region_idx, cp_dsmc, s=70, marker='x', facecolors='k', label='Cp DSMC')
        ax.scatter(region_idx, cp_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cp Distribution')
        # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
        
        ax.set_ylabel('Cp')
        ax.set_xlabel('Region Index')
        if title:
            ax.set_title(title)
        else:
            ax.set_title('Cp')
            
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = ["Times New Roman"]
        ax.grid(True, alpha=0.25)
        ax.legend()
        plt.show()

        # Cf plot
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111)
        ax.scatter(region_idx, cf_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cf Model')
        ax.scatter(region_idx, cf_dsmc, s=70, marker='x', facecolors='k', label='Cf DSMC')
        ax.scatter(region_idx, cf_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cf Distribution')
        # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
        
        ax.set_ylabel('Cf')
        ax.set_xlabel('Region Index')
        if title:
            ax.set_title(title)
        else:
            ax.set_title('Cf')
            
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = ["Times New Roman"]
        ax.grid(True, alpha=0.25)
        ax.legend()
        plt.show()

        # Ch plot
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111)
        ax.scatter(region_idx, ch_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='CH Model')
        ax.scatter(region_idx, ch_dsmc, s=70, marker='x', facecolors='k', label='CH DSMC')
        ax.scatter(region_idx, ch_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='CH Distribution')
        # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
        
        ax.set_ylabel('CH')
        ax.set_xlabel('Region Index')
        if title:
            ax.set_title(title)
        else:
            ax.set_title('CH')
            
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = ["Times New Roman"]
        ax.grid(True, alpha=0.25)
        ax.legend()
        plt.show()

        if all_in_one_plot:

                # Font Sizes
            plt.rcParams.update({
                'font.size': 28,          # General font size
                'axes.titlesize': 36,     # Title size
                'axes.labelsize': 32,     # Axis label (X and Y) size
                'xtick.labelsize': 24,    # Tick labels
                'ytick.labelsize': 24
            })
            
            fig = plt.figure(figsize=(30, 8))
            ax1 = fig.add_subplot(131)
            
            

            # Cp plot
            ax1.scatter(region_idx, cp_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cp Model')
            ax1.scatter(region_idx, cp_dsmc, s=70, marker='x', facecolors='k', label='Cp DSMC')
            ax1.scatter(region_idx, cp_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cp Distribution')
            # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
            
            ax1.set_ylabel('Cp')
            ax1.set_xlabel('Region Index')
            # ax1.legend()
                
            plt.rcParams["font.family"] = "serif"
            plt.rcParams["font.serif"] = ["Times New Roman"]
            ax1.grid(True, alpha=0.25)

            # Cf plot
            ax2 = fig.add_subplot(132)
            ax2.scatter(region_idx, cf_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Cf Model')
            ax2.scatter(region_idx, cf_dsmc, s=70, marker='x', facecolors='k', label='Cf DSMC')
            ax2.scatter(region_idx, cf_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='Cf Distribution')
            # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
            
            ax2.set_ylabel('Cf')
            ax2.set_xlabel('Region Index')
                
            plt.rcParams["font.family"] = "serif"
            plt.rcParams["font.serif"] = ["Times New Roman"]
            ax2.grid(True, alpha=0.25)

            # Ch plot
            ax3 = fig.add_subplot(133)
            ax3.scatter(region_idx, ch_model, s=70, marker='s', facecolors= 'none', edgecolors='k', label='Model')
            ax3.scatter(region_idx, ch_dsmc, s=70, marker='x', facecolors='k', label='DSMC')
            ax3.scatter(region_idx, ch_dist, s=70, marker='o', facecolors='w', edgecolors='b', label='CH Distribution')
            # ax.axhline(0.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
            
            ax3.set_ylabel('CH')
            ax3.set_xlabel('Region Index')
                
            plt.rcParams["font.family"] = "serif"
            plt.rcParams["font.serif"] = ["Times New Roman"]
            ax3.grid(True, alpha=0.25)
            ax3.legend(
                fontsize=16,            # Specifically set a smaller size for this legend
                loc='upper right',      # Force it to the top right corner
                frameon=True,           # Optional: adds a box around the legend
                edgecolor='k'           # Optional: makes the box border black
            )

            if title:
                fig.suptitle(title, fontsize=18)
            else:
                fig.suptitle('Cp, Cf, CH Comparison', fontsize=18)
            plt.show()

def plot_aerothermo_slides(moment_list, m, n_inf, u_inf, title=None, mode='triple'):
    """
    mode options: 
    'triple'     -> Cp, Cf, Ch in one row
    'dual'       -> Cp, Cf in one row
    'individual' -> Each plot in its own figure
    """
    # 1. Setup Constants and Global Slide Styling
    p_inf = 0.5 * m * n_inf * u_inf**2
    q_inf = 0.5 * m * n_inf * u_inf**3
    
    plt.rcParams.update({
        'font.size': 20,
        'axes.titlesize': 24,
        'axes.labelsize': 22,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18,
        'font.family': 'serif',
        'font.serif': ['Times New Roman']
    })

    colors = cm.plasma(np.linspace(0, 0.85, len(moment_list)))
    all_keys = [('pn', p_inf, 'Cp'), ('pt', p_inf, 'Cf'), ('e', q_inf, 'Ch')]

    # 2. Filter keys based on selected mode
    if mode == 'dual':
        plot_keys = all_keys[:2]
        fig_width = 18
    elif mode == 'triple':
        plot_keys = all_keys
        fig_width = 24
    else: # individual
        plot_keys = all_keys
        
    # 3. Plotting Logic
    if mode in ['dual', 'triple']:
        fig, axes = plt.subplots(1, len(plot_keys), figsize=(fig_width, 8), constrained_layout=True)
        if len(plot_keys) == 1: axes = [axes] # Ensure iterable
        
        _render_plots(axes, plot_keys, moment_list, colors, region_idx_len=len(moment_list[0]['pn_model']))
        if title: fig.suptitle(title, fontsize=28, fontweight='bold', y=1.05)
        plt.show()
        
    else: # Individual mode
        for prefix, norm, label in plot_keys:
            fig, ax = plt.subplots(figsize=(10, 8))
            _render_plots([ax], [(prefix, norm, label)], moment_list, colors, region_idx_len=len(moment_list[0]['pn_model']))
            if title: ax.set_title(f"{title}: {label}")
            plt.show()

def _render_plots(axes, plot_keys, moment_list, colors, region_idx_len):
    region_idx = np.arange(region_idx_len)
    
    for ax, (prefix, norm, label) in zip(axes, plot_keys):
        # Reference Data (Plot once)
        ref = moment_list[0]
        print(len(region_idx), len(ref[f'{prefix}_dsmc']), len(ref[f'{prefix}_dist']))
        ax.scatter(region_idx, ref[f'{prefix}_dsmc'] / norm, 
                   marker='x', color='black', s=150, linewidths=3, label='DSMC', zorder=5)
        ax.scatter(region_idx, ref[f'{prefix}_dist'] / norm, 
                   marker='o', facecolors='none', edgecolors='blue', s=150, linewidths=2, label='Dist', zorder=4)

        # Model Convergence
        for i, moments in enumerate(moment_list):
            n_terms = (i + 1)**3
            ax.scatter(region_idx, moments[f'{prefix}_model'] / norm, 
                       color=colors[i], marker='s', s=100, alpha=0.8, 
                       label=f'n={n_terms}', edgecolors='none')

        ax.set_ylabel(label, fontweight='bold')
        ax.set_xlabel('Region Index')
        ax.grid(True, alpha=0.3)
        
        # Legend setup - specific for the last plot in a row or for individual plots
        ax.legend(fontsize=14, loc='best', frameon=True, framealpha=0.9)
        
# def plot_aerothermo_convergence(moment_list, m, n_inf, u_inf, title=None):
#     # 1. Setup Constants and Colors
#     p_inf = 0.5 * m * n_inf * u_inf**2
#     q_inf = 0.5 * m * n_inf * u_inf**3
    
#     # Create a color gradient based on the number of models
#     colors = cm.viridis(np.linspace(0, 0.8, len(moment_list)))
    
#     # 2. Initialize Figure (1 row, 3 columns)
#     fig, axes = plt.subplots(1, 3, figsize=(20, 6), constrained_layout=True)
#     labels = ['Cp', 'Cf', 'Ch']
#     keys = [('pn', p_inf), ('pt', p_inf), ('e', q_inf)] # (prefix, normalizer)

#     # 3. Plot Reference Data (DSMC and Dist) from the first entry only
#     ref = moment_list[0]
#     region_idx = np.arange(len(ref['pn_model']))
    
#     for ax, (prefix, norm) in zip(axes, keys):
#         ax.scatter(region_idx, ref[f'{prefix}_dsmc'] / norm, 
#                    marker='x', color='black', s=80, label='DSMC', zorder=5)
#         ax.scatter(region_idx, ref[f'{prefix}_dist'] / norm, 
#                    marker='o', facecolors='none', edgecolors='blue', s=80, label='Dist', zorder=4)

#     # 4. Loop through models to show convergence
#     for i, moments in enumerate(moment_list):
#         n_terms = (i + 1)**2
#         color = colors[i]
        
#         for ax, (prefix, norm) in zip(axes, keys):
#             val_model = moments[f'{prefix}_model'] / norm
#             ax.scatter(region_idx, val_model, color=color, marker='s', 
#                        s=40, alpha=0.7, label=f'Model (n={n_terms})')

#     # 5. Global Formatting
#     for i, ax in enumerate(axes):
#         ax.set_ylabel(labels[i], fontsize=14, fontweight='bold')
#         ax.set_xlabel('Region Index', fontsize=12)
#         ax.grid(True, alpha=0.3, linestyle='--')
        
#         # Only show legend on the last plot to avoid cluttering
#         if i == 2:
#             ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)

#     if title:
#         fig.suptitle(title, fontsize=18, family='serif')
    
#     plt.show()

In [ ]:
# KN_eq_0p01_path = "/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p01_3D/0/"
# KN_eq_0p1_path = "/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_0p1_3D/0/"
# KN_eq_1p0_path = "/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_1p0_3D/0/"
# KN_eq_5p0_path = "/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_5p0_3D/0/"
# KN_eq_10p0_path = "/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/KN_eq_10p0_3D/0/"
# metadata_path = Path("/Volumes/SherlockScratch/ARGON_CYLINDER_2D_DISTRIBUTIONS/validation_case_argon_circle/argon_circle_0_metadata.txt")

# print("Loading data...")
# # region_cache_0p01, region_df = load_run_data_3D(KN_eq_0p01_path, metadata_path)
# # print("Loaded KN=0.01 data")
# # region_cache_0p1, _ = load_run_data_3D(KN_eq_0p1_path, metadata_path)
# # print("Loaded KN=0.1 data")
# # region_cache_1p0, _ = load_run_data_3D(KN_eq_1p0_path, metadata_path)
# # print("Loaded KN=1.0 data")
# region_cache_5p0, region_df = load_run_data_3D(KN_eq_5p0_path, metadata_path)
# print("Loaded KN=5.0 data")
# region_cache_10p0, _ = load_run_data_3D(KN_eq_10p0_path, metadata_path)
# print("Loaded KN=10.0 data")

import pickle
# # To SAVE
# print("Saving data to pickle files...")
# with open('KN_eq_5p0_data.pkl', 'wb') as f:
#     pickle.dump((region_cache_5p0, region_df), f)
# print("Saved KN=5.0 data")

# with open('KN_eq_10p0_data.pkl', 'wb') as f:
#     pickle.dump((region_cache_10p0, region_df), f)
# print("Saved KN=10.0 data")

# with open('KN_eq_1p0_data.pkl', 'wb') as f:
#     pickle.dump((region_cache_1p0, region_df), f)
# print("Saved KN=1.0 data")

# try:
#     with open('KN_eq_0p1_data.pkl', 'rb') as f:
#         region_cache_0p1, region_df = pickle.load(f)
#     print("Loaded successfully")

#     # print("Sanity Check: Printing Data")
#     # print(region_cache_0p1[15]["data"])
#     # print("Sanity Check 2:")
#     # print(type(region_cache_0p1))
# except Exception as e:
#     print("Errooorrr:", e)

# To LOAD
# with open('KN_eq_0p01_data.pkl', 'rb') as f:
#     region_cache_0p01, region_df = pickle.load(f)

# with open('KN_eq_1p0_data.pkl', 'wb') as f:
#     pickle.dump((region_cache_1p0, region_df), f)

# with open('KN_eq_0p01_data.pkl', 'rb') as f:
#     region_cache_0p01, region_df = pickle.load(f)
# print("Kn = 0.01: Loaded successfully")

# with open('KN_eq_0p1_data.pkl', 'rb') as f:
#     region_cache_0p1, region_df = pickle.load(f)
# print("Kn = 0.1: Loaded successfully")

import os
import yaml

# Resolved through data/registry.yaml (docs/CLEANUP_TODO.md, Phase 3 step 10) --
# never a hard-coded path, even though today only one entry is verified.
_registry_path = Path("../../../data/registry.yaml")  # relative to this notebook's directory
_registry_entry = yaml.safe_load(_registry_path.read_text())["kn_eq_1p0_data"]
assert _registry_entry["status"] == "verified", _registry_entry["status"]
_kn_eq_1p0_path = os.path.expanduser(_registry_entry["local_path"])

with open(_kn_eq_1p0_path, 'rb') as f:
    region_cache_1p0, region_df = pickle.load(f)
print("Kn = 1.0: Loaded successfully")

# with open('KN_eq_5p0_data.pkl', 'rb') as f:
#     region_cache_5p0, region_df = pickle.load(f)
# print("Kn = 5.0: Loaded successfully")

# with open('KN_eq_10p0_data.pkl', 'rb') as f:
#     region_cache_10p0, region_df = pickle.load(f)
# print("Kn = 10.0: Loaded successfully")


import tqdm

N_TERMS = 6
moment_0p01_list = []
moment_0p1_list = []
moment_1p0_list = []
moment_5p0_list = []
moment_10p0_list = []
# moment_5p0_list_DEBUG = []
# moment_10p0_list_DEBUG = []

for n in tqdm.tqdm(range(1, N_TERMS + 1)):
    # moments_0p01 = calculate_and_compare_moments(region_cache_0p01, region_df, n_terms=n, verbose=False)
    # moments_0p1 = calculate_and_compare_moments(region_cache_0p1, region_df, n_terms=n, verbose=False)
    moments_1p0 = calculate_and_compare_moments(region_cache_1p0, region_df, n_terms=n, verbose=False)
    # moments_5p0 = calculate_and_compare_moments(region_cache_5p0, region_df, n_terms=n, verbose=False)
    # moments_10p0 = calculate_and_compare_moments(region_cache_10p0, region_df, n_terms=n, verbose=False)
    # moment_0p01_list.append(moments_0p01)
    # moment_0p1_list.append(moments_0p1)
    moment_1p0_list.append(moments_1p0)
    # moment_5p0_list.append(moments_5p0)
    # moment_10p0_list.append(moments_10p0)
    # moment_5p0_list_DEBUG.append(moments_5p0_DEBUG)
    # moment_10p0_list_DEBUG.append(moments_10p0_DEBUG)
#     # plot_cp_cf_ch(moments_0p01, m=6.36e-26, n_inf=4.247e20, u_inf=2634.1, title=f"Argon Cylinder, KN=0.01, n_terms={n}", all_in_one_plot=True)
#     # plot_cp_cf_ch(moments_0p1, m=6.36e-26, n_inf=4.247e19, u_inf=2634.1, title=f"KN=0.1, n_terms={n}", all_in_one_plot=True)  
#     # plot_cp_cf_ch(moments_1p0, m=6.36e-26, n_inf=4.247e18, u_inf=2634.1, title=f"KN=1.0, n_terms={n}", all_in_one_plot=True)

# plot_cp_cf_ch_all(moment_list, m=6.36e-26, n_inf=4.247e20, u_inf=2634.1, title=f"TEST", all_in_one_plot=False)
# plot_aerothermo_slides(moment_0p01_list, m=6.36e-26, n_inf=4.247e20, u_inf=2634.1, title="Convergence of Cp and Cf, KN=0.01", mode='dual')
# plot_aerothermo_slides(moment_0p1_list, m=6.36e-26, n_inf=4.247e19, u_inf=2634.1, title="Convergence of Cp and Cf, KN=0.1", mode='dual')
# plot_aerothermo_slides(moment_1p0_list, m=6.36e-26, n_inf=4.247e18, u_inf=2634.1, title="Convergence of Cp and Cf, KN=1.0", mode='dual')

# plot_aerothermo_slides(moment_0p01_list, m=6.36e-26, n_inf=4.247e20, u_inf=2634.1, title="Convergence, KN=0.01", mode='triple')
# plot_aerothermo_slides(moment_0p1_list, m=6.36e-26, n_inf=4.247e19, u_inf=2634.1, title="Convergence, KN=0.1", mode='triple')
plot_aerothermo_slides(moment_1p0_list, m=6.36e-26, n_inf=4.247e18, u_inf=2634.1, title="Convergence, KN=1.0", mode='triple')
# print(moment_5p0_list)
# print(moment_10p0_list)
# plot_aerothermo_slides(moment_5p0_list, m=6.36e-26, n_inf=8.494e17, u_inf=2634.1, title="Convergence, KN=5.0", mode='triple')
# plot_aerothermo_slides(moment_10p0_list, m=6.36e-26, n_inf=4.247e17, u_inf=2634.1, title="Convergence, KN=10.0", mode='triple')